#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from joblib import dump, load

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

#### Loading the Prepared Forecasting Datasets

In [2]:
processed_dir = Path("../data/processed")
models_dir = Path("../models")
artifacts_dir = Path("../artifacts")

models_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)

daily_df = pd.read_csv(
    processed_dir / "daily_reservations.csv",
    parse_dates=["arrival_datetime"]
)

train_df = pd.read_csv(
    processed_dir / "train_daily.csv",
    parse_dates=["arrival_datetime"]
)

test_df = pd.read_csv(
    processed_dir / "test_daily.csv",
    parse_dates=["arrival_datetime"]
)

train_features_df = pd.read_csv(
    processed_dir / "train_features.csv",
    parse_dates=["arrival_datetime"]
)

test_features_df = pd.read_csv(
    processed_dir / "test_features.csv",
    parse_dates=["arrival_datetime"]
)

target_column = "scheduled_reservations"
date_column = "arrival_datetime"
forecast_horizon = 7

feature_columns = [
    column
    for column in train_features_df.columns
    if column not in [date_column, target_column]
]

print("Forecasting Datasets Loaded Successfully")

print("\nOriginal Daily Series:")
print(f"Complete Dataset: {daily_df.shape}")
print(f"Training Dataset: {train_df.shape}")
print(f"Testing Dataset: {test_df.shape}")

print("\nFeature-Engineered Datasets:")
print(f"Training Dataset: {train_features_df.shape}")
print(f"Testing Dataset: {test_features_df.shape}")
print(f"Predictor Features: {len(feature_columns)}")

print("\nForecasting Configuration:")
print(f"Target Variable: {target_column}")
print(f"Forecast Horizon: {forecast_horizon} days")

print("\nArtifact Directories:")
print(f"Models Directory: {models_dir}")
print(f"Artifacts Directory: {artifacts_dir}")

Forecasting Datasets Loaded Successfully

Original Daily Series:
Complete Dataset: (549, 2)
Training Dataset: (439, 2)
Testing Dataset: (110, 2)

Feature-Engineered Datasets:
Training Dataset: (405, 26)
Testing Dataset: (110, 26)
Predictor Features: 24

Forecasting Configuration:
Target Variable: scheduled_reservations
Forecast Horizon: 7 days

Artifact Directories:
Models Directory: ..\models
Artifacts Directory: ..\artifacts


#### Key Observations
- **Successful Dataset Loading:** The original daily series contains 439 training and 110 testing observations, while the feature-engineered datasets contain 405 training and 110 testing observations.
- **Modeling Configuration Confirmed:** All 24 engineered predictors are available, the forecasting horizon is seven days, and the directories for saving models and artifacts have been created.

#### Validating Forecasting Dataset Integrity

In [3]:
validation_results = {
    "Original Training Dates Sorted": (
        train_df[date_column].is_monotonic_increasing
    ),
    "Original Testing Dates Sorted": (
        test_df[date_column].is_monotonic_increasing
    ),
    "Feature Training Dates Sorted": (
        train_features_df[date_column].is_monotonic_increasing
    ),
    "Feature Testing Dates Sorted": (
        test_features_df[date_column].is_monotonic_increasing
    ),
    "Chronological Separation": (
        train_df[date_column].max() < test_df[date_column].min()
    ),
    "Feature Columns Consistent": (
        train_features_df.columns.equals(test_features_df.columns)
    ),
    "No Missing Training Features": (
        train_features_df[feature_columns].notna().all().all()
    ),
    "No Missing Training Targets": (
        train_df[target_column].notna().all()
    ),
    "Training Targets Preserved": (
        train_features_df.set_index(date_column)[target_column].equals(
            train_df.set_index(date_column)[target_column].loc[
                train_features_df[date_column]
            ].set_axis(train_features_df[date_column])
        )
    ),
    "Original Daily Series Reconstructed": (
        pd.concat(
            [train_df, test_df],
            ignore_index=True
        ).equals(daily_df)
    )
}

print("Forecasting Dataset Integrity Validation:")
for check, result in validation_results.items():
    print(f"{check}: {result}")

print("\nOverall Validation:")
print(f"All Checks Passed: {all(validation_results.values())}")

Forecasting Dataset Integrity Validation:
Original Training Dates Sorted: True
Original Testing Dates Sorted: True
Feature Training Dates Sorted: True
Feature Testing Dates Sorted: True
Chronological Separation: True
Feature Columns Consistent: True
No Missing Training Features: True
No Missing Training Targets: True
Training Targets Preserved: True
Original Daily Series Reconstructed: True

Overall Validation:
All Checks Passed: True


#### Key Observations
- **Dataset Integrity Confirmed:** All original and feature-engineered datasets passed validation, with consistent feature structures, preserved target values, and correctly ordered dates.
- **Chronological Separation Maintained:** The training and testing periods remain independent, and the complete daily series can be reconstructed without losing observations.

#### Defining Expanding-Window Validation Folds

In [4]:
initial_train_size = 280
validation_step = forecast_horizon

validation_folds = []

for train_end in range(
    initial_train_size,
    len(train_df) - forecast_horizon + 1,
    validation_step
):
    validation_end = train_end + forecast_horizon

    validation_folds.append({
        "fold": len(validation_folds) + 1,
        "train_start": train_df[date_column].iloc[0],
        "train_end": train_df[date_column].iloc[train_end - 1],
        "validation_start": train_df[date_column].iloc[train_end],
        "validation_end": train_df[date_column].iloc[validation_end - 1],
        "train_size": train_end,
        "validation_size": forecast_horizon
    })

validation_folds_df = pd.DataFrame(validation_folds)

print("Expanding-Window Validation Configuration:")
print(f"Initial Training Size: {initial_train_size} days")
print(f"Forecast Horizon: {forecast_horizon} days")
print(f"Validation Step: {validation_step} days")
print(f"Total Validation Folds: {len(validation_folds_df)}")

print("\nValidation Fold Summary:")
print(validation_folds_df.to_string(index=False))

print("\nValidation Integrity:")
print(
    "All Validation Windows Have Seven Days:",
    validation_folds_df["validation_size"].eq(forecast_horizon).all()
)

print(
    "All Training Periods Precede Validation:",
    (
        validation_folds_df["train_end"]
        < validation_folds_df["validation_start"]
    ).all()
)

print(
    "Validation Remains Within Training Period:",
    (
        validation_folds_df["validation_end"]
        <= train_df[date_column].max()
    ).all()
)

Expanding-Window Validation Configuration:
Initial Training Size: 280 days
Forecast Horizon: 7 days
Validation Step: 7 days
Total Validation Folds: 22

Validation Fold Summary:
 fold train_start  train_end validation_start validation_end  train_size  validation_size
    1  2017-07-01 2018-04-06       2018-04-07     2018-04-13         280                7
    2  2017-07-01 2018-04-13       2018-04-14     2018-04-20         287                7
    3  2017-07-01 2018-04-20       2018-04-21     2018-04-27         294                7
    4  2017-07-01 2018-04-27       2018-04-28     2018-05-04         301                7
    5  2017-07-01 2018-05-04       2018-05-05     2018-05-11         308                7
    6  2017-07-01 2018-05-11       2018-05-12     2018-05-18         315                7
    7  2017-07-01 2018-05-18       2018-05-19     2018-05-25         322                7
    8  2017-07-01 2018-05-25       2018-05-26     2018-06-01         329                7
    9  2017-0

#### Key Observations
- **Expanding-Window Validation Established:** The framework contains 22 validation folds, beginning with 280 training observations and expanding by seven days after each forecast.
- **Forecasting Integrity Confirmed:** Every validation window contains exactly seven consecutive days, with training observations strictly preceding their corresponding validation periods.
- **Test Data Isolation Maintained:** All validation folds remain within the original training period, preserving the final 110-day testing period for independent evaluation.

#### Defining Forecasting Evaluation Metrics

In [5]:
def evaluate_forecast(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if y_true.shape != y_pred.shape:
        raise ValueError("Actual and predicted values must have matching shapes.")

    if not np.isfinite(y_true).all() or not np.isfinite(y_pred).all():
        raise ValueError("Actual and predicted values must be finite.")

    y_pred = np.maximum(y_pred, 0)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    actual_total = np.abs(y_true).sum()

    wape = (
        np.abs(y_true - y_pred).sum() / actual_total * 100
        if actual_total > 0
        else np.nan
    )

    r2 = (
        r2_score(y_true, y_pred)
        if len(y_true) >= 2
        else np.nan
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape,
        "R2": r2
    }


print("Forecasting Evaluation Function Defined Successfully")

print("\nValidation Example:")

actual = [50, 60, 70, 80, 90, 100, 110]
predicted = [55, 65, 75, 85, 95, 105, 115]

example_metrics = evaluate_forecast(actual, predicted)

for metric, value in example_metrics.items():
    print(f"{metric}: {value:.4f}")

Forecasting Evaluation Function Defined Successfully

Validation Example:
MAE: 5.0000
RMSE: 5.0000
WAPE: 6.2500
R2: 0.9375


#### Key Observations
- **Evaluation Framework Validated:** The reusable function successfully calculates MAE, RMSE, WAPE, and R², with an expected MAE and RMSE of 5.00 for the sample predictions.
- **Consistent Forecast Evaluation:** The function validates prediction shapes and finite values while enforcing nonnegative forecasts for reservation counts.

#### Developing Baseline Forecasting Models

In [6]:
def generate_baseline_forecasts(history, horizon=7):
    history = np.asarray(history, dtype=float)

    if len(history) < 7:
        raise ValueError("At least seven historical observations are required.")

    forecasts = {
        "Naive": np.repeat(history[-1], horizon),
        "Seasonal Naive": np.resize(history[-7:], horizon),
        "Historical Moving Average": np.repeat(
            history[-7:].mean(), horizon
        )
    }

    return forecasts


baseline_predictions = []
baseline_results = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    forecasts = generate_baseline_forecasts(
        history,
        horizon=forecast_horizon
    )

    for model_name, predicted in forecasts.items():
        metrics = evaluate_forecast(actual, predicted)

        baseline_results.append({
            "Model": model_name,
            "Fold": int(fold["fold"]),
            **metrics
        })

        for step, (actual_value, predicted_value) in enumerate(
            zip(actual, predicted),
            start=1
        ):
            baseline_predictions.append({
                "Model": model_name,
                "Fold": int(fold["fold"]),
                "Forecast_Step": step,
                "Actual": actual_value,
                "Predicted": max(float(predicted_value), 0.0)
            })

baseline_results_df = pd.DataFrame(baseline_results)
baseline_predictions_df = pd.DataFrame(baseline_predictions)

print("Baseline Forecasting Models Evaluated Successfully")

print("\nValidation Summary:")
print(f"Models Evaluated: {baseline_results_df['Model'].nunique()}")
print(f"Validation Folds: {baseline_results_df['Fold'].nunique()}")
print(f"Total Model-Fold Evaluations: {len(baseline_results_df)}")
print(f"Total Forecast Records: {len(baseline_predictions_df)}")

print("\nAverage Validation Metrics:")
baseline_summary_df = (
    baseline_results_df
    .groupby("Model")[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .sort_values("MAE")
    .round(4)
)

print(baseline_summary_df.to_string())

Baseline Forecasting Models Evaluated Successfully

Validation Summary:
Models Evaluated: 3
Validation Folds: 22
Total Model-Fold Evaluations: 66
Total Forecast Records: 462

Average Validation Metrics:
                               MAE     RMSE    WAPE      R2
Model                                                      
Historical Moving Average  25.6809  30.4130  27.867 -0.2786
Seasonal Naive             31.5714  38.5895  34.259 -1.6664
Naive                      37.2143  43.6073  40.322 -3.6521


#### Key Observations
- **Historical Moving Average Performs Best:** The seven-day moving average achieves the lowest MAE (25.68), RMSE (30.41), and WAPE (27.87%) among the three baseline models.
- **Simple Forecasting Methods Show Limitations:** Seasonal Naive (MAE 31.57) outperforms Naive (MAE 37.21), but all three models produce negative average fold-level R² values, indicating difficulty capturing short-term reservation fluctuations.

#### Developing Simple Exponential Smoothing Forecasts

In [7]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

ses_results = []
ses_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = SimpleExpSmoothing(
        history,
        initialization_method="estimated"
    )

    fitted_model = model.fit(
        optimized=True
    )

    predicted = np.maximum(
        np.asarray(fitted_model.forecast(forecast_horizon)),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    ses_results.append({
        "Model": "Simple Exponential Smoothing",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        ses_predictions.append({
            "Model": "Simple Exponential Smoothing",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

ses_results_df = pd.DataFrame(ses_results)
ses_predictions_df = pd.DataFrame(ses_predictions)

print("Simple Exponential Smoothing Evaluated Successfully")

print("\nValidation Summary:")
print(f"Validation Folds: {len(ses_results_df)}")
print(f"Total Forecast Records: {len(ses_predictions_df)}")

print("\nAverage Validation Metrics:")
print(
    ses_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_string()
)

print("\nComparison with Best Baseline:")
best_baseline_mae = baseline_summary_df["MAE"].min()
ses_mae = ses_results_df["MAE"].mean()

print(f"Historical Moving Average MAE: {best_baseline_mae:.4f}")
print(f"Simple Exponential Smoothing MAE: {ses_mae:.4f}")
print(f"MAE Improvement: {best_baseline_mae - ses_mae:.4f}")

Simple Exponential Smoothing Evaluated Successfully

Validation Summary:
Validation Folds: 22
Total Forecast Records: 154

Average Validation Metrics:
MAE     25.6880
RMSE    30.1320
WAPE    27.8631
R2      -0.2483

Comparison with Best Baseline:
Historical Moving Average MAE: 25.6809
Simple Exponential Smoothing MAE: 25.6880
MAE Improvement: -0.0071


#### Key Observations
- **Comparable Forecasting Accuracy:** Simple Exponential Smoothing achieves an MAE of 25.69 and RMSE of 30.13, performing almost identically to the Historical Moving Average baseline.
- **Limited Improvement Over Baselines:** Although SES slightly reduces RMSE, its MAE remains marginally higher than the best baseline, indicating no meaningful improvement in average absolute forecasting accuracy.

#### Consolidating Model Validation Results

In [9]:
model_results_df = pd.concat(
    [baseline_results_df, ses_results_df],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [baseline_predictions_df, ses_predictions_df],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Consolidated Model Results Initialized Successfully")

print("\nCurrent Model Comparison:")
print(model_comparison_df.round(4).to_string(index=False))

print("\nConsolidated Results Summary:")
print(f"Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Fold-Level Evaluations: {len(model_results_df)}")
print(f"Individual Forecasts: {len(model_predictions_df)}")

print("\nSaved Artifacts:")
print("model_validation_results.csv")
print("model_validation_predictions.csv")
print("model_comparison.csv")

Consolidated Model Results Initialized Successfully

Current Model Comparison:
                       Model     MAE    RMSE    WAPE      R2  Validation_Folds
   Historical Moving Average 25.6809 30.4130 27.8670 -0.2786                22
Simple Exponential Smoothing 25.6880 30.1320 27.8631 -0.2483                22
              Seasonal Naive 31.5714 38.5895 34.2590 -1.6664                22
                       Naive 37.2143 43.6073 40.3220 -3.6521                22

Consolidated Results Summary:
Models Evaluated: 4
Fold-Level Evaluations: 88
Individual Forecasts: 616

Saved Artifacts:
model_validation_results.csv
model_validation_predictions.csv
model_comparison.csv


#### Developing Holt's Linear Trend Forecasts

In [10]:
from statsmodels.tsa.holtwinters import Holt

holt_results = []
holt_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = Holt(
        history,
        exponential=False,
        damped_trend=True,
        initialization_method="estimated"
    )

    fitted_model = model.fit(
        optimized=True
    )

    predicted = np.maximum(
        np.asarray(fitted_model.forecast(forecast_horizon)),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    holt_results.append({
        "Model": "Holt Linear Trend",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        holt_predictions.append({
            "Model": "Holt Linear Trend",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

holt_results_df = pd.DataFrame(holt_results)
holt_predictions_df = pd.DataFrame(holt_predictions)

print("Holt's Linear Trend Evaluated Successfully")

print("\nValidation Summary:")
print(f"Validation Folds: {len(holt_results_df)}")
print(f"Total Forecast Records: {len(holt_predictions_df)}")

print("\nAverage Validation Metrics:")
print(
    holt_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_string()
)

print("\nComparison with Best Existing Model:")

best_existing_model = model_comparison_df.iloc[0]
holt_mae = holt_results_df["MAE"].mean()

print(f"Best Existing Model: {best_existing_model['Model']}")
print(f"Best Existing MAE: {best_existing_model['MAE']:.4f}")
print(f"Holt Linear Trend MAE: {holt_mae:.4f}")
print(f"MAE Improvement: {best_existing_model['MAE'] - holt_mae:.4f}")

Holt's Linear Trend Evaluated Successfully

Validation Summary:
Validation Folds: 22
Total Forecast Records: 154

Average Validation Metrics:
MAE     25.6826
RMSE    30.1377
WAPE    27.8564
R2      -0.2489

Comparison with Best Existing Model:
Best Existing Model: Historical Moving Average
Best Existing MAE: 25.6809
Holt Linear Trend MAE: 25.6826
MAE Improvement: -0.0017


In [11]:
model_results_df = pd.concat([model_results_df, holt_results_df], ignore_index=True)
model_predictions_df = pd.concat([model_predictions_df, holt_predictions_df], ignore_index=True)

In [12]:
display(model_comparison_df)

,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Historical Moving Average,25.680891,30.413028,27.866991,-0.278649,22
1,Simple Exponential Smoothing,25.688002,30.131954,27.863129,-0.248315,22
2,Seasonal Naive,31.571429,38.589536,34.258952,-1.666351,22
3,Naive,37.214286,43.607331,40.322012,-3.652148,22


#### Developing Holt-Winters Exponential Smoothing Forecasts

In [13]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

holt_winters_results = []
holt_winters_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = ExponentialSmoothing(
        history,
        trend="add",
        damped_trend=True,
        seasonal="add",
        seasonal_periods=7,
        initialization_method="estimated"
    )

    fitted_model = model.fit(optimized=True)

    predicted = np.maximum(
        np.asarray(fitted_model.forecast(forecast_horizon)),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    holt_winters_results.append({
        "Model": "Holt-Winters",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        holt_winters_predictions.append({
            "Model": "Holt-Winters",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

holt_winters_results_df = pd.DataFrame(holt_winters_results)
holt_winters_predictions_df = pd.DataFrame(holt_winters_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Holt-Winters"
        ],
        holt_winters_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Holt-Winters"
        ],
        holt_winters_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Holt-Winters Exponential Smoothing Evaluated Successfully")

print("\nAverage Validation Metrics:")
display(
    holt_winters_results_df[
        ["MAE", "RMSE", "WAPE", "R2"]
    ].mean().round(4).to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

Holt-Winters Exponential Smoothing Evaluated Successfully

Average Validation Metrics:


,Value
MAE,26.0370
RMSE,30.7946
WAPE,28.4497
R2,-0.3557



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
1,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
2,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
3,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
4,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
5,Naive,37.2143,43.6073,40.3220,-3.6521,22



Total Models Evaluated: 6


#### Key Observations
- **Historical Moving Average Remains the Best Model:** It achieves the lowest MAE (25.68), narrowly outperforming Holt's Linear Trend (25.68) and Simple Exponential Smoothing (25.69).
- **Weekly Seasonality Provides Limited Improvement:** Holt-Winters achieves an MAE of 26.04, indicating that incorporating additive weekly seasonality and trend does not improve forecasting accuracy over the simpler approaches.

#### Developing ARIMA Forecasts

In [14]:
from statsmodels.tsa.arima.model import ARIMA

arima_results = []
arima_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = ARIMA(
        history,
        order=(1, 1, 1),
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_model = model.fit()

    predicted = np.maximum(
        np.asarray(fitted_model.forecast(steps=forecast_horizon)),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    arima_results.append({
        "Model": "ARIMA",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        arima_predictions.append({
            "Model": "ARIMA",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

arima_results_df = pd.DataFrame(arima_results)
arima_predictions_df = pd.DataFrame(arima_predictions)

model_results_df = pd.concat(
    [
        model_results_df[model_results_df["Model"] != "ARIMA"],
        arima_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[model_predictions_df["Model"] != "ARIMA"],
        arima_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("ARIMA Evaluated Successfully")

print("\nARIMA Average Validation Metrics:")
display(
    arima_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

ARIMA Evaluated Successfully

ARIMA Average Validation Metrics:


,Value
MAE,25.6696
RMSE,30.1731
WAPE,27.8351
R2,-0.2533



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
1,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
2,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
3,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
4,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
5,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
6,Naive,37.2143,43.6073,40.3220,-3.6521,22



Total Models Evaluated: 7


#### Key Observations
- **ARIMA Achieves the Lowest Forecasting Error:** ARIMA records the best MAE (25.67) and WAPE (27.84%) among the seven evaluated models, marginally outperforming the Historical Moving Average baseline.
- **Performance Improvements Remain Limited:** ARIMA's MAE improvement over the strongest baseline is only 0.01 reservations, while its RMSE (30.17) remains slightly higher than that of Simple Exponential Smoothing (30.13).

#### Developing Seasonal ARIMA Forecasts

In [15]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_results = []
sarima_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:train_end + forecast_horizon
    ].to_numpy()

    model = SARIMAX(
        history,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 7),
        trend="n",
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_model = model.fit(
        disp=False,
        maxiter=100
    )

    predicted = np.maximum(
        np.asarray(
            fitted_model.forecast(steps=forecast_horizon)
        ),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    sarima_results.append({
        "Model": "SARIMA",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        sarima_predictions.append({
            "Model": "SARIMA",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

sarima_results_df = pd.DataFrame(sarima_results)
sarima_predictions_df = pd.DataFrame(sarima_predictions)

model_results_df = pd.concat(
    [
        model_results_df[model_results_df["Model"] != "SARIMA"],
        sarima_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "SARIMA"
        ],
        sarima_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("SARIMA Evaluated Successfully")

print("\nSARIMA Average Validation Metrics:")
display(
    sarima_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

SARIMA Evaluated Successfully

SARIMA Average Validation Metrics:


,Value
MAE,25.3638
RMSE,30.1642
WAPE,27.6425
R2,-0.2890



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
6,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
7,Naive,37.2143,43.6073,40.3220,-3.6521,22



Total Models Evaluated: 8


#### Key Observations
- **SARIMA Achieves the Best Forecasting Accuracy:** SARIMA records the lowest MAE (25.36) and WAPE (27.64%), outperforming ARIMA (25.67) and the Historical Moving Average baseline (25.68).
- **Seasonal Modeling Provides a Modest Improvement:** Incorporating weekly seasonal autoregressive relationships reduces average forecasting error, although SARIMA's RMSE (30.16) and negative average R² indicate that substantial prediction errors remain.

#### Developing SARIMAX Forecasts with Calendar Features

In [16]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

calendar_df = train_df[[date_column, target_column]].copy()

calendar_df["month"] = calendar_df[date_column].dt.month
calendar_df["day_of_week"] = calendar_df[date_column].dt.dayofweek
calendar_df["is_weekend"] = (
    calendar_df["day_of_week"] >= 5
).astype(int)

calendar_exog = pd.get_dummies(
    calendar_df[["month", "day_of_week"]],
    columns=["month", "day_of_week"],
    drop_first=True,
    dtype=float
)

calendar_exog["is_weekend"] = calendar_df["is_weekend"].astype(float)

sarimax_results = []
sarimax_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end = int(fold["train_size"])
    validation_end = train_end + forecast_horizon

    history = train_df[target_column].iloc[:train_end].to_numpy()

    actual = train_df[target_column].iloc[
        train_end:validation_end
    ].to_numpy()

    exog_train = calendar_exog.iloc[:train_end]
    exog_validation = calendar_exog.iloc[
        train_end:validation_end
    ]

    model = SARIMAX(
        history,
        exog=exog_train,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 7),
        trend="n",
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_model = model.fit(
        disp=False,
        maxiter=100
    )

    predicted = np.maximum(
        np.asarray(
            fitted_model.forecast(
                steps=forecast_horizon,
                exog=exog_validation
            )
        ),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    sarimax_results.append({
        "Model": "SARIMAX",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        sarimax_predictions.append({
            "Model": "SARIMAX",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

sarimax_results_df = pd.DataFrame(sarimax_results)
sarimax_predictions_df = pd.DataFrame(sarimax_predictions)

model_results_df = pd.concat(
    [
        model_results_df[model_results_df["Model"] != "SARIMAX"],
        sarimax_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "SARIMAX"
        ],
        sarimax_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("SARIMAX Evaluated Successfully")

print("\nSARIMAX Average Validation Metrics:")
display(
    sarimax_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

C:\Users\user\AppData\Local\Temp\ipykernel_204\276153991.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar_df["day_of_week"] = calendar_df[date_column].dt.dayofweek
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
d:\Projects\ML and Data Science\StaySens

SARIMAX Evaluated Successfully

SARIMAX Average Validation Metrics:


d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


,Value
MAE,37.4344
RMSE,43.5902
WAPE,41.1529
R2,-4.4552



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
6,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
7,Naive,37.2143,43.6073,40.3220,-3.6521,22
8,SARIMAX,37.4344,43.5902,41.1529,-4.4552,22



Total Models Evaluated: 9


#### Key Observations
- **SARIMA Remains the Best-Performing Model:** SARIMA achieves the lowest MAE (25.36) and WAPE (27.64%) among all nine evaluated models.
- **Calendar-Based SARIMAX Underperforms:** SARIMAX records an MAE of 37.43 and RMSE of 43.59, indicating that the selected calendar predictors and model configuration do not improve forecasting accuracy.

#### Developing Machine Learning Models

In [17]:
from sklearn.linear_model import LinearRegression

linear_results = []
linear_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = LinearRegression()
    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()

    metrics = evaluate_forecast(actual, predicted)

    linear_results.append({
        "Model": "Linear Regression",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        linear_predictions.append({
            "Model": "Linear Regression",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

linear_results_df = pd.DataFrame(linear_results)
linear_predictions_df = pd.DataFrame(linear_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Linear Regression"
        ],
        linear_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Linear Regression"
        ],
        linear_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Linear Regression Evaluated Successfully")

print("\nLinear Regression Average Validation Metrics:")
display(
    linear_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")

Linear Regression Evaluated Successfully

Linear Regression Average Validation Metrics:


,Value
MAE,27.8522
RMSE,33.3249
WAPE,30.4829
R2,-0.7453



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
6,Linear Regression,27.8522,33.3249,30.4829,-0.7453,22
7,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
8,Naive,37.2143,43.6073,40.3220,-3.6521,22
9,SARIMAX,37.4344,43.5902,41.1529,-4.4552,22



Total Models Evaluated: 10


#### Key Observations
- **Linear Regression Underperforms Statistical Models:** Linear Regression achieves an MAE of 27.85 and RMSE of 33.32, indicating weaker forecasting accuracy than the leading statistical approaches.
- **SARIMA Retains the Best Performance:** SARIMA remains the strongest model with an MAE of 25.36, outperforming Linear Regression by approximately 2.49 reservations.

#### Developing Ridge Regression Forecasts

In [18]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ridge_results = []
ridge_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    ridge_results.append({
        "Model": "Ridge Regression",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        ridge_predictions.append({
            "Model": "Ridge Regression",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

ridge_results_df = pd.DataFrame(ridge_results)
ridge_predictions_df = pd.DataFrame(ridge_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Ridge Regression"
        ],
        ridge_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Ridge Regression"
        ],
        ridge_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Ridge Regression Evaluated Successfully")

print("\nRidge Regression Average Validation Metrics:")
display(
    ridge_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

Ridge Regression Evaluated Successfully

Ridge Regression Average Validation Metrics:


,Value
MAE,26.2196
RMSE,31.6286
WAPE,28.5679
R2,-0.4703



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
6,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
7,Linear Regression,27.8522,33.3249,30.4829,-0.7453,22
8,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22
9,Naive,37.2143,43.6073,40.3220,-3.6521,22



Total Models Evaluated: 11
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Key Observations
- **Ridge Regression Improves Linear Modeling:** Ridge achieves an MAE of 26.22 and RMSE of 31.63, outperforming Linear Regression (MAE 27.85), suggesting that L2 regularization improves forecasting stability.
- **SARIMA Maintains the Lowest Forecasting Error:** SARIMA remains the best-performing model with an MAE of 25.36, outperforming Ridge Regression by approximately 0.86 reservations.

#### Developing ElasticNet Regression Forecasts

In [19]:
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

elasticnet_results = []
elasticnet_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = make_pipeline(
        StandardScaler(),
        ElasticNet(
            alpha=0.1,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=RANDOM_STATE
        )
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    elasticnet_results.append({
        "Model": "ElasticNet Regression",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        elasticnet_predictions.append({
            "Model": "ElasticNet Regression",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

elasticnet_results_df = pd.DataFrame(elasticnet_results)
elasticnet_predictions_df = pd.DataFrame(elasticnet_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "ElasticNet Regression"
        ],
        elasticnet_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "ElasticNet Regression"
        ],
        elasticnet_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("ElasticNet Regression Evaluated Successfully")

print("\nElasticNet Average Validation Metrics:")
display(
    elasticnet_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

ElasticNet Regression Evaluated Successfully

ElasticNet Average Validation Metrics:


,Value
MAE,25.9838
RMSE,31.3737
WAPE,28.2804
R2,-0.4330



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
6,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
7,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
8,Linear Regression,27.8522,33.3249,30.4829,-0.7453,22
9,Seasonal Naive,31.5714,38.5895,34.2590,-1.6664,22



Total Models Evaluated: 12
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Key Observations
- **ElasticNet Leads Linear Regression Models:** ElasticNet achieves an MAE of 25.98 and RMSE of 31.37, outperforming both Ridge Regression (MAE 26.22) and Linear Regression (MAE 27.85).
- **SARIMA Remains the Best Overall Model:** SARIMA retains the lowest MAE (25.36), outperforming ElasticNet by approximately 0.62 reservations.

#### Developing Decision Tree Regression Forecasts

In [20]:
from sklearn.tree import DecisionTreeRegressor

decision_tree_results = []
decision_tree_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = DecisionTreeRegressor(
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=RANDOM_STATE
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    decision_tree_results.append({
        "Model": "Decision Tree",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        decision_tree_predictions.append({
            "Model": "Decision Tree",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

decision_tree_results_df = pd.DataFrame(decision_tree_results)
decision_tree_predictions_df = pd.DataFrame(decision_tree_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Decision Tree"
        ],
        decision_tree_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Decision Tree"
        ],
        decision_tree_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Decision Tree Regression Evaluated Successfully")

print("\nDecision Tree Average Validation Metrics:")
display(
    decision_tree_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

Decision Tree Regression Evaluated Successfully

Decision Tree Average Validation Metrics:


,Value
MAE,28.1437
RMSE,35.2922
WAPE,30.8076
R2,-1.2217



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
6,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
7,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
8,Linear Regression,27.8522,33.3249,30.4829,-0.7453,22
9,Decision Tree,28.1437,35.2922,30.8076,-1.2217,22



Total Models Evaluated: 13
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Key Observations
- **Decision Tree Shows Weaker Forecasting Accuracy:** Decision Tree achieves an MAE of 28.14 and RMSE of 35.29, underperforming ElasticNet (MAE 25.98) and Ridge Regression (MAE 26.22).
- **SARIMA Remains the Best Overall Model:** SARIMA maintains the lowest MAE (25.36) and WAPE (27.64%), outperforming Decision Tree by approximately 2.78 reservations.

#### Developing Random Forest Regression Forecasts

In [21]:
from sklearn.ensemble import RandomForestRegressor

random_forest_results = []
random_forest_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    random_forest_results.append({
        "Model": "Random Forest",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        random_forest_predictions.append({
            "Model": "Random Forest",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

random_forest_results_df = pd.DataFrame(random_forest_results)
random_forest_predictions_df = pd.DataFrame(random_forest_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Random Forest"
        ],
        random_forest_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Random Forest"
        ],
        random_forest_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Random Forest Regression Evaluated Successfully")

print("\nRandom Forest Average Validation Metrics:")
display(
    random_forest_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

Random Forest Regression Evaluated Successfully

Random Forest Average Validation Metrics:


,Value
MAE,26.3328
RMSE,31.5232
WAPE,28.7778
R2,-0.5129



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
6,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
7,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
8,Random Forest,26.3328,31.5232,28.7778,-0.5129,22
9,Linear Regression,27.8522,33.3249,30.4829,-0.7453,22



Total Models Evaluated: 14
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Key Observations
- **Random Forest Improves Tree-Based Forecasting:** Random Forest achieves an MAE of 26.33 and RMSE of 31.52, outperforming Decision Tree (MAE 28.14), indicating that ensemble averaging improves forecasting accuracy.
- **SARIMA Remains the Best Overall Model:** SARIMA retains the lowest MAE (25.36) and WAPE (27.64%), outperforming Random Forest by approximately 0.97 reservations.

#### Developing Extra Trees Regression Forecasts

In [22]:
from sklearn.ensemble import ExtraTreesRegressor

extra_trees_results = []
extra_trees_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = ExtraTreesRegressor(
        n_estimators=200,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    extra_trees_results.append({
        "Model": "Extra Trees",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        extra_trees_predictions.append({
            "Model": "Extra Trees",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

extra_trees_results_df = pd.DataFrame(extra_trees_results)
extra_trees_predictions_df = pd.DataFrame(extra_trees_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Extra Trees"
        ],
        extra_trees_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Extra Trees"
        ],
        extra_trees_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Extra Trees Regression Evaluated Successfully")

print("\nExtra Trees Average Validation Metrics:")
display(
    extra_trees_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

Extra Trees Regression Evaluated Successfully

Extra Trees Average Validation Metrics:


,Value
MAE,25.7271
RMSE,31.0040
WAPE,28.1458
R2,-0.4313



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
6,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
7,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
8,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
9,Random Forest,26.3328,31.5232,28.7778,-0.5129,22



Total Models Evaluated: 15
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Key Observations
- **Extra Trees Leads Machine Learning Models:** Extra Trees achieves an MAE of 25.73 and RMSE of 31.00, outperforming Random Forest (MAE 26.33) and ElasticNet Regression (MAE 25.98).
- **SARIMA Remains the Best Overall Model:** SARIMA maintains the lowest MAE (25.36) and WAPE (27.64%), narrowly outperforming Extra Trees by approximately 0.36 reservations.

#### Developing Histogram Gradient Boosting Forecasts

In [23]:
from sklearn.ensemble import HistGradientBoostingRegressor

hist_gb_results = []
hist_gb_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        max_depth=5,
        min_samples_leaf=10,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=RANDOM_STATE
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    hist_gb_results.append({
        "Model": "HistGradientBoosting",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        hist_gb_predictions.append({
            "Model": "HistGradientBoosting",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

hist_gb_results_df = pd.DataFrame(hist_gb_results)
hist_gb_predictions_df = pd.DataFrame(hist_gb_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "HistGradientBoosting"
        ],
        hist_gb_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "HistGradientBoosting"
        ],
        hist_gb_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("HistGradientBoosting Evaluated Successfully")

print("\nHistGradientBoosting Average Validation Metrics:")
display(
    hist_gb_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

HistGradientBoosting Evaluated Successfully

HistGradientBoosting Average Validation Metrics:


,Value
MAE,28.8388
RMSE,36.2107
WAPE,31.5725
R2,-1.3323



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
6,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
7,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
8,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
9,Random Forest,26.3328,31.5232,28.7778,-0.5129,22



Total Models Evaluated: 16
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Developing XGBoost Regression Forecasts

In [24]:
from xgboost import XGBRegressor

xgboost_results = []
xgboost_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    xgboost_results.append({
        "Model": "XGBoost",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        xgboost_predictions.append({
            "Model": "XGBoost",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

xgboost_results_df = pd.DataFrame(xgboost_results)
xgboost_predictions_df = pd.DataFrame(xgboost_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "XGBoost"
        ],
        xgboost_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "XGBoost"
        ],
        xgboost_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("XGBoost Regression Evaluated Successfully")

print("\nXGBoost Average Validation Metrics:")
display(
    xgboost_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

XGBoost Regression Evaluated Successfully

XGBoost Average Validation Metrics:


,Value
MAE,28.6343
RMSE,35.2061
WAPE,31.3215
R2,-1.0500



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
6,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
7,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
8,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
9,Random Forest,26.3328,31.5232,28.7778,-0.5129,22



Total Models Evaluated: 17
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Key Observations
- **XGBoost Shows Limited Forecasting Improvement:** XGBoost achieves an MAE of 28.63 and RMSE of 35.21, slightly outperforming HistGradientBoosting (MAE 28.84) but trailing Extra Trees (MAE 25.73).
- **SARIMA Remains the Best Overall Model:** SARIMA maintains the lowest MAE (25.36) and WAPE (27.64%), outperforming XGBoost by approximately 3.27 reservations.

#### Developing LightGBM Regression Forecasts

In [25]:
from lightgbm import LGBMRegressor

lightgbm_results = []
lightgbm_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=15,
        max_depth=5,
        min_child_samples=15,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=2.0,
        verbosity=-1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    lightgbm_results.append({
        "Model": "LightGBM",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        lightgbm_predictions.append({
            "Model": "LightGBM",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

lightgbm_results_df = pd.DataFrame(lightgbm_results)
lightgbm_predictions_df = pd.DataFrame(lightgbm_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "LightGBM"
        ],
        lightgbm_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "LightGBM"
        ],
        lightgbm_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("LightGBM Regression Evaluated Successfully")

print("\nLightGBM Average Validation Metrics:")
display(
    lightgbm_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

LightGBM Regression Evaluated Successfully

LightGBM Average Validation Metrics:


,Value
MAE,27.5591
RMSE,34.3502
WAPE,30.2391
R2,-1.0738



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
6,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
7,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
8,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
9,Random Forest,26.3328,31.5232,28.7778,-0.5129,22



Total Models Evaluated: 18
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Key Observations
- **LightGBM Leads Gradient Boosting Models:** LightGBM achieves an MAE of 27.56 and RMSE of 34.35, outperforming XGBoost (MAE 28.63) and HistGradientBoosting (MAE 28.84).
- **SARIMA Maintains the Best Forecasting Accuracy:** SARIMA remains the best-performing model with an MAE of 25.36 and WAPE of 27.64%, outperforming LightGBM by approximately 2.20 reservations.

#### Developing CatBoost Regression Forecasts

In [28]:
%pip install --upgrade --force-reinstall --no-deps catboost

  Using cached catboost-1.2.10-cp311-cp311-win_amd64.whl (100.2 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device


[notice] A new release of pip is available: 23.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Developing Multi-Layer Perceptron Forecasts

In [29]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

mlp_results = []
mlp_predictions = []

for _, fold in validation_folds_df.iterrows():
    train_end_date = fold["train_end"]
    validation_start = fold["validation_start"]
    validation_end = fold["validation_end"]

    fold_train = train_features_df[
        train_features_df[date_column] <= train_end_date
    ]

    fold_validation = train_features_df[
        train_features_df[date_column].between(
            validation_start,
            validation_end
        )
    ]

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]

    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    model = make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="adam",
            alpha=1.0,
            learning_rate_init=0.001,
            max_iter=1000,
            early_stopping=False,
            random_state=RANDOM_STATE
        )
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )

    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    mlp_results.append({
        "Model": "MLP Regression",
        "Fold": int(fold["fold"]),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        mlp_predictions.append({
            "Model": "MLP Regression",
            "Fold": int(fold["fold"]),
            "Forecast_Step": step,
            "Actual": actual_value,
            "Predicted": float(predicted_value)
        })

mlp_results_df = pd.DataFrame(mlp_results)
mlp_predictions_df = pd.DataFrame(mlp_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "MLP Regression"
        ],
        mlp_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "MLP Regression"
        ],
        mlp_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("MLP Regression Evaluated Successfully")

print("\nMLP Regression Average Validation Metrics:")
display(
    mlp_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reac

MLP Regression Evaluated Successfully

MLP Regression Average Validation Metrics:


d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


,Value
MAE,32.3778
RMSE,38.5249
WAPE,35.6406
R2,-1.6273



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
1,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
2,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
3,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
4,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
5,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
6,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
7,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
8,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22
9,Random Forest,26.3328,31.5232,28.7778,-0.5129,22



Total Models Evaluated: 19
Best Performing Model: SARIMA
Best Model MAE: 25.3638


#### Key Observations
- **MLP Regression Shows Limited Forecasting Accuracy:** MLP achieves an MAE of 32.38 and RMSE of 38.52, underperforming Extra Trees (MAE 25.73) and ElasticNet Regression (MAE 25.98).
- **SARIMA Remains the Best Overall Model:** SARIMA maintains the lowest MAE (25.36) and WAPE (27.64%), outperforming MLP Regression by approximately 7.01 reservations.

#### Checking TensorFlow Availability

In [31]:
import importlib.util

tensorflow_available = importlib.util.find_spec("tensorflow") is not None
torch_available = importlib.util.find_spec("torch") is not None

print(f"TensorFlow Available: {tensorflow_available}")
print(f"PyTorch Available: {torch_available}")

if tensorflow_available:
    print("\nTensorFlow is available for LSTM and GRU modeling.")
elif torch_available:
    print("\nPyTorch is available as an alternative for LSTM and GRU modeling.")
else:
    print("\nNeither framework is installed. Additional installation would be required.")

TensorFlow Available: False
PyTorch Available: True

PyTorch is available as an alternative for LSTM and GRU modeling.


#### Developing LSTM Forecasts

In [32]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(2)

class LSTMForecaster(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )
        self.output = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return self.output(hidden[-1])

lstm_results = []
lstm_predictions = []

lookback = 28
epochs = 100

for _, fold in validation_folds_df.iterrows():
    fold_number = int(fold["fold"])

    fold_train = train_df[
        train_df[date_column] <= fold["train_end"]
    ]

    fold_validation = train_df[
        train_df[date_column].between(
            fold["validation_start"],
            fold["validation_end"]
        )
    ]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    train_values = fold_train[target_column].to_numpy(dtype=np.float32)
    actual = fold_validation[target_column].to_numpy(dtype=np.float32)

    scaler = StandardScaler()
    scaled_train = scaler.fit_transform(
        train_values.reshape(-1, 1)
    ).ravel()

    X_sequences = []
    y_sequences = []

    for start in range(
        len(scaled_train) - lookback - forecast_horizon + 1
    ):
        X_sequences.append(
            scaled_train[start:start + lookback]
        )
        y_sequences.append(
            scaled_train[
                start + lookback:
                start + lookback + forecast_horizon
            ]
        )

    X_tensor = torch.tensor(
        np.array(X_sequences),
        dtype=torch.float32
    ).unsqueeze(-1)

    y_tensor = torch.tensor(
        np.array(y_sequences),
        dtype=torch.float32
    )

    torch.manual_seed(RANDOM_STATE)

    model = LSTMForecaster(hidden_size=16)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=0.001
    )
    criterion = nn.MSELoss()

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_tensor)
        loss = criterion(predictions, y_tensor)
        loss.backward()
        optimizer.step()

    model.eval()

    input_sequence = torch.tensor(
        scaled_train[-lookback:],
        dtype=torch.float32
    ).reshape(1, lookback, 1)

    with torch.no_grad():
        scaled_forecast = (
            model(input_sequence)
            .numpy()
            .reshape(-1, 1)
        )

    predicted = np.maximum(
        scaler.inverse_transform(scaled_forecast).ravel(),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    lstm_results.append({
        "Model": "LSTM",
        "Fold": fold_number,
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        lstm_predictions.append({
            "Model": "LSTM",
            "Fold": fold_number,
            "Forecast_Step": step,
            "Actual": float(actual_value),
            "Predicted": float(predicted_value)
        })

lstm_results_df = pd.DataFrame(lstm_results)
lstm_predictions_df = pd.DataFrame(lstm_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "LSTM"
        ],
        lstm_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "LSTM"
        ],
        lstm_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("LSTM Evaluated Successfully")

print("\nLSTM Average Validation Metrics:")
display(
    lstm_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

LSTM Evaluated Successfully

LSTM Average Validation Metrics:


,Value
MAE,25.0399
RMSE,30.4504
WAPE,27.0267
R2,-0.3142



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,LSTM,25.0399,30.4504,27.0267,-0.3142,22
1,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
2,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
3,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
4,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
5,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
6,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
7,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
8,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22
9,Ridge Regression,26.2196,31.6286,28.5679,-0.4703,22



Total Models Evaluated: 20
Best Performing Model: LSTM
Best Model MAE: 25.0399


#### Key Observations
- **LSTM Achieves the Lowest Forecasting Error:** LSTM records an MAE of 25.04 and WAPE of 27.03%, outperforming SARIMA by approximately 0.32 reservations in average absolute error.
- **Neural Forecasting Shows Competitive Performance:** LSTM ranks first among 20 models by MAE, although SARIMA retains a slightly lower RMSE (30.16 versus 30.45), indicating that LSTM does not outperform every model across all evaluation metrics.

#### Developing GRU Forecasts

In [34]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(2)

class GRUForecaster(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )
        self.output = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.output(hidden[-1])

gru_results = []
gru_predictions = []

lookback = 28
epochs = 100

for _, fold in validation_folds_df.iterrows():
    fold_number = int(fold["fold"])

    fold_train = train_df[
        train_df[date_column] <= fold["train_end"]
    ]

    fold_validation = train_df[
        train_df[date_column].between(
            fold["validation_start"],
            fold["validation_end"]
        )
    ]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    train_values = fold_train[target_column].to_numpy(dtype=np.float32)
    actual = fold_validation[target_column].to_numpy(dtype=np.float32)

    scaler = StandardScaler()
    scaled_train = scaler.fit_transform(
        train_values.reshape(-1, 1)
    ).ravel()

    X_sequences = []
    y_sequences = []

    for start in range(
        len(scaled_train) - lookback - forecast_horizon + 1
    ):
        X_sequences.append(
            scaled_train[start:start + lookback]
        )
        y_sequences.append(
            scaled_train[
                start + lookback:
                start + lookback + forecast_horizon
            ]
        )

    X_tensor = torch.tensor(
        np.array(X_sequences),
        dtype=torch.float32
    ).unsqueeze(-1)

    y_tensor = torch.tensor(
        np.array(y_sequences),
        dtype=torch.float32
    )

    torch.manual_seed(RANDOM_STATE)

    model = GRUForecaster(hidden_size=16)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=0.001
    )

    criterion = nn.MSELoss()

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_tensor)
        loss = criterion(predictions, y_tensor)
        loss.backward()
        optimizer.step()

    model.eval()

    input_sequence = torch.tensor(
        scaled_train[-lookback:],
        dtype=torch.float32
    ).reshape(1, lookback, 1)

    with torch.no_grad():
        scaled_forecast = (
            model(input_sequence)
            .numpy()
            .reshape(-1, 1)
        )

    predicted = np.maximum(
        scaler.inverse_transform(scaled_forecast).ravel(),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    gru_results.append({
        "Model": "GRU",
        "Fold": fold_number,
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        gru_predictions.append({
            "Model": "GRU",
            "Fold": fold_number,
            "Forecast_Step": step,
            "Actual": float(actual_value),
            "Predicted": float(predicted_value)
        })

gru_results_df = pd.DataFrame(gru_results)
gru_predictions_df = pd.DataFrame(gru_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "GRU"
        ],
        gru_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "GRU"
        ],
        gru_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("GRU Evaluated Successfully")

print("\nGRU Average Validation Metrics:")
display(
    gru_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

GRU Evaluated Successfully

GRU Average Validation Metrics:


,Value
MAE,25.6713
RMSE,30.8085
WAPE,27.8006
R2,-0.3373



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,LSTM,25.0399,30.4504,27.0267,-0.3142,22
1,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
2,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
3,GRU,25.6713,30.8085,27.8006,-0.3373,22
4,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
5,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
6,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
7,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
8,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
9,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22



Total Models Evaluated: 21
Best Performing Model: LSTM
Best Model MAE: 25.0399


#### Key Observations
- **GRU Delivers Competitive Forecasting Accuracy:** GRU achieves an MAE of 25.67 and WAPE of 27.80%, ranking fourth among 21 models but performing slightly worse than SARIMA and LSTM.
- **LSTM Maintains the Lowest Forecasting Error:** LSTM remains the leading model with an MAE of 25.04, outperforming GRU by approximately 0.63 reservations per forecasted day.

#### Developing CNN Forecasts

In [35]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(2)

class CNNForecaster(nn.Module):
    def __init__(self, lookback=28):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.Conv1d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(7)
        )

        self.output = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7, 32),
            nn.ReLU(),
            nn.Linear(32, forecast_horizon)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.features(x)
        return self.output(x)

cnn_results = []
cnn_predictions = []

lookback = 28
epochs = 100

for _, fold in validation_folds_df.iterrows():
    fold_number = int(fold["fold"])

    fold_train = train_df[
        train_df[date_column] <= fold["train_end"]
    ]

    fold_validation = train_df[
        train_df[date_column].between(
            fold["validation_start"],
            fold["validation_end"]
        )
    ]

    if len(fold_validation) != forecast_horizon:
        raise ValueError(
            "Validation window must contain seven observations."
        )

    train_values = fold_train[target_column].to_numpy(
        dtype=np.float32
    )

    actual = fold_validation[target_column].to_numpy(
        dtype=np.float32
    )

    scaler = StandardScaler()

    scaled_train = scaler.fit_transform(
        train_values.reshape(-1, 1)
    ).ravel()

    X_sequences = []
    y_sequences = []

    for start in range(
        len(scaled_train) - lookback - forecast_horizon + 1
    ):
        X_sequences.append(
            scaled_train[start:start + lookback]
        )

        y_sequences.append(
            scaled_train[
                start + lookback:
                start + lookback + forecast_horizon
            ]
        )

    X_tensor = torch.tensor(
        np.array(X_sequences),
        dtype=torch.float32
    ).unsqueeze(-1)

    y_tensor = torch.tensor(
        np.array(y_sequences),
        dtype=torch.float32
    )

    torch.manual_seed(RANDOM_STATE)

    model = CNNForecaster(lookback=lookback)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=0.001
    )

    criterion = nn.MSELoss()

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()

        predictions = model(X_tensor)

        loss = criterion(predictions, y_tensor)

        loss.backward()
        optimizer.step()

    model.eval()

    input_sequence = torch.tensor(
        scaled_train[-lookback:],
        dtype=torch.float32
    ).reshape(1, lookback, 1)

    with torch.no_grad():
        scaled_forecast = (
            model(input_sequence)
            .numpy()
            .reshape(-1, 1)
        )

    predicted = np.maximum(
        scaler.inverse_transform(
            scaled_forecast
        ).ravel(),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    cnn_results.append({
        "Model": "CNN",
        "Fold": fold_number,
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        cnn_predictions.append({
            "Model": "CNN",
            "Fold": fold_number,
            "Forecast_Step": step,
            "Actual": float(actual_value),
            "Predicted": float(predicted_value)
        })

cnn_results_df = pd.DataFrame(cnn_results)
cnn_predictions_df = pd.DataFrame(cnn_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "CNN"
        ],
        cnn_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "CNN"
        ],
        cnn_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("CNN Evaluated Successfully")

print("\nCNN Average Validation Metrics:")

display(
    cnn_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")

display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(
    f"\nTotal Models Evaluated: "
    f"{model_results_df['Model'].nunique()}"
)

print(
    f"Best Performing Model: {best_model['Model']}"
)

print(
    f"Best Model MAE: {best_model['MAE']:.4f}"
)

CNN Evaluated Successfully

CNN Average Validation Metrics:


,Value
MAE,27.1719
RMSE,32.9701
WAPE,29.3547
R2,-0.5681



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,LSTM,25.0399,30.4504,27.0267,-0.3142,22
1,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
2,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
3,GRU,25.6713,30.8085,27.8006,-0.3373,22
4,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
5,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
6,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
7,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
8,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22
9,Holt-Winters,26.0370,30.7946,28.4497,-0.3557,22



Total Models Evaluated: 22
Best Performing Model: LSTM
Best Model MAE: 25.0399


#### Key Observations
- **CNN Demonstrates Moderate Forecasting Accuracy:** CNN achieves an MAE of 27.17 and WAPE of 29.35%, ranking 13th among 22 models and performing less accurately than LSTM and GRU.
- **LSTM Remains the Best-Performing Model:** LSTM maintains the lowest MAE of 25.04, outperforming CNN by approximately 2.13 reservations per forecasted day, although SARIMA retains a lower RMSE.

#### Developing Voting Regression Forecasts

In [36]:
from sklearn.ensemble import VotingRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

voting_results = []
voting_predictions = []

for _, fold in validation_folds_df.iterrows():
    fold_number = int(fold["fold"])

    fold_train = train_features_df[
        train_features_df[date_column] <= fold["train_end"]
    ]
    fold_validation = train_features_df[
        train_features_df[date_column].between(
            fold["validation_start"],
            fold["validation_end"]
        )
    ]

    if len(fold_validation) != forecast_horizon:
        raise ValueError("Validation window must contain seven observations.")

    X_train_fold = fold_train[feature_columns]
    y_train_fold = fold_train[target_column]
    X_validation_fold = fold_validation[feature_columns]
    y_validation_fold = fold_validation[target_column]

    model = VotingRegressor(
        estimators=[
            (
                "extra_trees",
                ExtraTreesRegressor(
                    n_estimators=200,
                    max_depth=8,
                    min_samples_split=10,
                    min_samples_leaf=4,
                    max_features=0.8,
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )
            ),
            (
                "elasticnet",
                make_pipeline(
                    StandardScaler(),
                    ElasticNet(
                        alpha=0.1,
                        l1_ratio=0.5,
                        max_iter=10000,
                        random_state=RANDOM_STATE
                    )
                )
            ),
            (
                "ridge",
                make_pipeline(
                    StandardScaler(),
                    Ridge(alpha=10.0)
                )
            )
        ],
        n_jobs=1
    )

    model.fit(X_train_fold, y_train_fold)

    predicted = np.maximum(
        model.predict(X_validation_fold),
        0
    )
    actual = y_validation_fold.to_numpy()
    metrics = evaluate_forecast(actual, predicted)

    voting_results.append({
        "Model": "Voting Regressor",
        "Fold": fold_number,
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        voting_predictions.append({
            "Model": "Voting Regressor",
            "Fold": fold_number,
            "Forecast_Step": step,
            "Actual": float(actual_value),
            "Predicted": float(predicted_value)
        })

voting_results_df = pd.DataFrame(voting_results)
voting_predictions_df = pd.DataFrame(voting_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Voting Regressor"
        ],
        voting_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Voting Regressor"
        ],
        voting_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)
model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)
model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Voting Regression Evaluated Successfully")

print("\nVoting Regression Average Validation Metrics:")
display(
    voting_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

Voting Regression Evaluated Successfully

Voting Regression Average Validation Metrics:


,Value
MAE,25.7376
RMSE,30.9074
WAPE,28.0517
R2,-0.3848



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,LSTM,25.0399,30.4504,27.0267,-0.3142,22
1,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
2,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
3,GRU,25.6713,30.8085,27.8006,-0.3373,22
4,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
5,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
6,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
7,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
8,Voting Regressor,25.7376,30.9074,28.0517,-0.3848,22
9,ElasticNet Regression,25.9838,31.3737,28.2804,-0.4330,22



Total Models Evaluated: 23
Best Performing Model: LSTM
Best Model MAE: 25.0399


#### Key Observations
- **Voting Regression Delivers Competitive Forecasting Accuracy:** Voting Regression achieves an MAE of 25.74 and WAPE of 28.05%, ranking ninth among 23 models and performing closely to Extra Trees (MAE 25.73).
- **LSTM Maintains the Lowest Forecasting Error:** LSTM remains the leading model with an MAE of 25.04, outperforming Voting Regression by approximately 0.70 reservations per forecasted day.

#### Developing Weighted Averaging Forecasts

In [37]:
ensemble_weights = {
    "LSTM": 0.40,
    "SARIMA": 0.35,
    "Extra Trees": 0.25
}

weighted_models = list(ensemble_weights.keys())

ensemble_source_df = model_predictions_df[
    model_predictions_df["Model"].isin(weighted_models)
].copy()

expected_rows = len(validation_folds_df) * forecast_horizon

for model_name in weighted_models:
    model_rows = ensemble_source_df[
        ensemble_source_df["Model"] == model_name
    ]
    if len(model_rows) != expected_rows:
        raise ValueError(
            f"Incomplete validation predictions for {model_name}."
        )

if not np.isclose(sum(ensemble_weights.values()), 1.0):
    raise ValueError("Ensemble weights must sum to one.")

ensemble_source_df["Weighted_Prediction"] = (
    ensemble_source_df["Predicted"]
    * ensemble_source_df["Model"].map(ensemble_weights)
)

weighted_forecasts_df = (
    ensemble_source_df
    .groupby(["Fold", "Forecast_Step"], as_index=False)
    .agg(
        Actual=("Actual", "first"),
        Predicted=("Weighted_Prediction", "sum"),
        Model_Count=("Model", "nunique"),
        Actual_Count=("Actual", "nunique")
    )
    .sort_values(["Fold", "Forecast_Step"])
    .reset_index(drop=True)
)

if (
    len(weighted_forecasts_df) != expected_rows
    or not weighted_forecasts_df["Model_Count"].eq(len(weighted_models)).all()
    or not weighted_forecasts_df["Actual_Count"].eq(1).all()
):
    raise ValueError(
        "Missing model predictions or inconsistent actual values."
    )

weighted_forecasts_df["Predicted"] = np.maximum(
    weighted_forecasts_df["Predicted"],
    0
)

weighted_results = []

for fold_number, fold_data in weighted_forecasts_df.groupby("Fold"):
    actual = fold_data["Actual"].to_numpy()
    predicted = fold_data["Predicted"].to_numpy()

    metrics = evaluate_forecast(actual, predicted)

    weighted_results.append({
        "Model": "Weighted Averaging",
        "Fold": int(fold_number),
        **metrics
    })

weighted_results_df = pd.DataFrame(weighted_results)

weighted_predictions_df = weighted_forecasts_df[
    ["Fold", "Forecast_Step", "Actual", "Predicted"]
].copy()

weighted_predictions_df.insert(
    0,
    "Model",
    "Weighted Averaging"
)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Weighted Averaging"
        ],
        weighted_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Weighted Averaging"
        ],
        weighted_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Weighted Averaging Evaluated Successfully")

print("\nWeighted Averaging Average Validation Metrics:")
display(
    weighted_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

Weighted Averaging Evaluated Successfully

Weighted Averaging Average Validation Metrics:


,Value
MAE,24.8586
RMSE,29.5802
WAPE,27.0050
R2,-0.2174



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Weighted Averaging,24.8586,29.5802,27.0050,-0.2174,22
1,LSTM,25.0399,30.4504,27.0267,-0.3142,22
2,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
3,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
4,GRU,25.6713,30.8085,27.8006,-0.3373,22
5,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
6,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
7,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
8,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
9,Voting Regressor,25.7376,30.9074,28.0517,-0.3848,22



Total Models Evaluated: 24
Best Performing Model: Weighted Averaging
Best Model MAE: 24.8586


#### Key Observations
- **Weighted Averaging Achieves the Lowest Forecasting Error:** The ensemble records an MAE of 24.86, RMSE of 29.58, and WAPE of 27.01%, outperforming the individual LSTM, SARIMA, and Extra Trees models.
- **Combining Complementary Models Improves Forecast Stability:** Weighted Averaging reduces MAE by approximately 0.18 reservations compared with LSTM and achieves a lower RMSE, suggesting that combining statistical, neural, and tree-based forecasts can improve overall accuracy.

#### Developing Stacking Regression Forecasts

In [38]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

stacking_base_models = ["LSTM", "SARIMA", "Extra Trees"]
minimum_training_folds = 5

stacking_source_df = model_predictions_df[
    model_predictions_df["Model"].isin(stacking_base_models)
].copy()

expected_rows = len(validation_folds_df) * forecast_horizon

for model_name in stacking_base_models:
    model_rows = stacking_source_df[
        stacking_source_df["Model"] == model_name
    ]

    if len(model_rows) != expected_rows:
        raise ValueError(
            f"Incomplete validation predictions for {model_name}."
        )

if stacking_source_df.duplicated(
    ["Model", "Fold", "Forecast_Step"]
).any():
    raise ValueError("Duplicate base-model predictions detected.")

actual_consistency = stacking_source_df.groupby(
    ["Fold", "Forecast_Step"]
)["Actual"].nunique()

if not actual_consistency.eq(1).all():
    raise ValueError("Inconsistent actual values across base models.")

stacking_features_df = stacking_source_df.pivot(
    index=["Fold", "Forecast_Step"],
    columns="Model",
    values="Predicted"
).reset_index()

stacking_actual_df = (
    stacking_source_df
    .groupby(["Fold", "Forecast_Step"], as_index=False)
    .agg(Actual=("Actual", "first"))
)

stacking_features_df = (
    stacking_features_df
    .merge(
        stacking_actual_df,
        on=["Fold", "Forecast_Step"],
        validate="one_to_one"
    )
    .sort_values(["Fold", "Forecast_Step"])
    .reset_index(drop=True)
)

if (
    len(stacking_features_df) != expected_rows
    or stacking_features_df[stacking_base_models].isna().any().any()
):
    raise ValueError("Incomplete stacking feature matrix.")

stacking_results = []
stacking_predictions = []

fold_numbers = sorted(stacking_features_df["Fold"].unique())

for fold_number in fold_numbers[minimum_training_folds:]:
    meta_train = stacking_features_df[
        stacking_features_df["Fold"] < fold_number
    ]

    meta_validation = stacking_features_df[
        stacking_features_df["Fold"] == fold_number
    ]

    if meta_train["Fold"].nunique() < minimum_training_folds:
        raise ValueError("Insufficient historical meta-training folds.")

    if len(meta_validation) != forecast_horizon:
        raise ValueError(
            "Validation window must contain seven observations."
        )

    X_meta_train = meta_train[stacking_base_models]
    y_meta_train = meta_train["Actual"]

    X_meta_validation = meta_validation[stacking_base_models]
    actual = meta_validation["Actual"].to_numpy()

    meta_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=100.0)
    )

    meta_model.fit(X_meta_train, y_meta_train)

    predicted = np.maximum(
        meta_model.predict(X_meta_validation),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    stacking_results.append({
        "Model": "Stacking Regression",
        "Fold": int(fold_number),
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        stacking_predictions.append({
            "Model": "Stacking Regression",
            "Fold": int(fold_number),
            "Forecast_Step": step,
            "Actual": float(actual_value),
            "Predicted": float(predicted_value)
        })

stacking_results_df = pd.DataFrame(stacking_results)
stacking_predictions_df = pd.DataFrame(stacking_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Stacking Regression"
        ],
        stacking_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Stacking Regression"
        ],
        stacking_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Stacking Regression Evaluated Successfully")

print("\nStacking Regression Average Validation Metrics:")
display(
    stacking_results_df[["MAE", "RMSE", "WAPE", "R2"]]
    .mean()
    .round(4)
    .to_frame("Value")
)

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

common_folds = sorted(stacking_results_df["Fold"].unique())

common_fold_comparison_df = (
    model_results_df[
        model_results_df["Fold"].isin(common_folds)
    ]
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .query("Validation_Folds == @len_common_folds")
    if False else None
)

len_common_folds = len(common_folds)

common_fold_comparison_df = (
    model_results_df[
        model_results_df["Fold"].isin(common_folds)
    ]
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
)

common_fold_comparison_df = (
    common_fold_comparison_df[
        common_fold_comparison_df["Validation_Folds"] == len_common_folds
    ]
    .sort_values("MAE")
    .reset_index(drop=True)
)

print(
    f"\nFair Comparison Across {len_common_folds} "
    "Common Validation Folds:"
)
display(common_fold_comparison_df.round(4))

common_best_model = common_fold_comparison_df.iloc[0]

print(f"\nBest Model on Common Folds: {common_best_model['Model']}")
print(f"Best Common-Fold MAE: {common_best_model['MAE']:.4f}")

Stacking Regression Evaluated Successfully

Stacking Regression Average Validation Metrics:


,Value
MAE,28.4294
RMSE,33.0453
WAPE,30.8223
R2,-0.3195



Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Weighted Averaging,24.8586,29.5802,27.0050,-0.2174,22
1,LSTM,25.0399,30.4504,27.0267,-0.3142,22
2,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
3,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
4,GRU,25.6713,30.8085,27.8006,-0.3373,22
5,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
6,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
7,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
8,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22
9,Voting Regressor,25.7376,30.9074,28.0517,-0.3848,22



Total Models Evaluated: 25
Best Performing Model: Weighted Averaging
Best Model MAE: 24.8586

Fair Comparison Across 17 Common Validation Folds:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,LSTM,26.4736,32.4454,28.2715,-0.1922,17
1,Weighted Averaging,27.2306,32.5717,29.3352,-0.2569,17
2,GRU,27.5682,33.1001,29.5645,-0.2780,17
3,Holt-Winters,27.9138,33.4529,30.2443,-0.3646,17
4,SARIMA,27.9205,33.2533,30.1777,-0.3359,17
5,Voting Regressor,28.0915,33.8520,30.3370,-0.4434,17
6,ElasticNet Regression,28.1315,33.9949,30.3064,-0.4518,17
7,Historical Moving Average,28.2953,33.5743,30.4448,-0.3396,17
8,ARIMA,28.3057,33.2110,30.4278,-0.2972,17
9,Ridge Regression,28.3729,34.1784,30.6052,-0.4818,17



Best Model on Common Folds: LSTM
Best Common-Fold MAE: 26.4736


#### Key Observations
- **Stacking Regression Does Not Improve Forecasting Accuracy:** Stacking achieves an MAE of 28.43 across 17 validation folds, underperforming LSTM (26.47) and Weighted Averaging (27.23) when evaluated over the same periods.
- **Model Rankings Vary Across Validation Periods:** Weighted Averaging leads the complete 22-fold comparison with an MAE of 24.86, while LSTM leads the 17-fold comparison with an MAE of 26.47, highlighting differences in performance across forecasting periods.

#### Tuning LSTM Hyperparameters

In [39]:
from itertools import product
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

torch.set_num_threads(2)

lstm_param_grid = list(product(
    [14, 28],
    [16, 32],
    [0.001, 0.003]
))

lstm_tuning_results = []

for lookback, hidden_size, learning_rate in lstm_param_grid:
    fold_metrics = []

    for _, fold in validation_folds_df.iterrows():
        fold_train = train_df[
            train_df[date_column] <= fold["train_end"]
        ]

        fold_validation = train_df[
            train_df[date_column].between(
                fold["validation_start"],
                fold["validation_end"]
            )
        ]

        train_values = fold_train[target_column].to_numpy(
            dtype=np.float32
        )
        actual = fold_validation[target_column].to_numpy(
            dtype=np.float32
        )

        if len(actual) != forecast_horizon:
            raise ValueError("Validation window must contain seven observations.")

        scaler = StandardScaler()
        scaled_train = scaler.fit_transform(
            train_values.reshape(-1, 1)
        ).ravel()

        X_sequences = []
        y_sequences = []

        for start in range(
            len(scaled_train) - lookback - forecast_horizon + 1
        ):
            X_sequences.append(
                scaled_train[start:start + lookback]
            )
            y_sequences.append(
                scaled_train[
                    start + lookback:
                    start + lookback + forecast_horizon
                ]
            )

        X_tensor = torch.tensor(
            np.array(X_sequences),
            dtype=torch.float32
        ).unsqueeze(-1)

        y_tensor = torch.tensor(
            np.array(y_sequences),
            dtype=torch.float32
        )

        torch.manual_seed(RANDOM_STATE)

        model = LSTMForecaster(hidden_size=hidden_size)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=0.001
        )

        criterion = nn.MSELoss()
        model.train()

        for epoch in range(100):
            optimizer.zero_grad()
            predictions = model(X_tensor)
            loss = criterion(predictions, y_tensor)
            loss.backward()
            optimizer.step()

        model.eval()

        input_sequence = torch.tensor(
            scaled_train[-lookback:],
            dtype=torch.float32
        ).reshape(1, lookback, 1)

        with torch.no_grad():
            scaled_forecast = (
                model(input_sequence)
                .numpy()
                .reshape(-1, 1)
            )

        predicted = np.maximum(
            scaler.inverse_transform(scaled_forecast).ravel(),
            0
        )

        metrics = evaluate_forecast(actual, predicted)
        fold_metrics.append(metrics)

    fold_metrics_df = pd.DataFrame(fold_metrics)

    lstm_tuning_results.append({
        "Lookback": lookback,
        "Hidden_Size": hidden_size,
        "Learning_Rate": learning_rate,
        "MAE": fold_metrics_df["MAE"].mean(),
        "RMSE": fold_metrics_df["RMSE"].mean(),
        "WAPE": fold_metrics_df["WAPE"].mean(),
        "R2": fold_metrics_df["R2"].mean(),
        "Validation_Folds": len(fold_metrics_df)
    })

    print(
        f"Completed: Lookback={lookback}, "
        f"Hidden_Size={hidden_size}, "
        f"Learning_Rate={learning_rate}"
    )

lstm_tuning_df = (
    pd.DataFrame(lstm_tuning_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

lstm_tuning_df.to_csv(
    artifacts_dir / "lstm_hyperparameter_tuning.csv",
    index=False
)

print("\nLSTM Hyperparameter Tuning Results:")
display(lstm_tuning_df.round(4))

best_lstm_config = lstm_tuning_df.iloc[0]

print("\nBest LSTM Configuration:")
print(f"Lookback: {int(best_lstm_config['Lookback'])}")
print(f"Hidden Units: {int(best_lstm_config['Hidden_Size'])}")
print(f"Learning Rate: {best_lstm_config['Learning_Rate']}")
print(f"Best Tuned LSTM MAE: {best_lstm_config['MAE']:.4f}")

print("\nCurrent Overall Best Model:")
best_model = model_comparison_df.iloc[0]
print(f"Model: {best_model['Model']}")
print(f"MAE: {best_model['MAE']:.4f}")

Completed: Lookback=14, Hidden_Size=16, Learning_Rate=0.001
Completed: Lookback=14, Hidden_Size=16, Learning_Rate=0.003
Completed: Lookback=14, Hidden_Size=32, Learning_Rate=0.001
Completed: Lookback=14, Hidden_Size=32, Learning_Rate=0.003
Completed: Lookback=28, Hidden_Size=16, Learning_Rate=0.001
Completed: Lookback=28, Hidden_Size=16, Learning_Rate=0.003
Completed: Lookback=28, Hidden_Size=32, Learning_Rate=0.001
Completed: Lookback=28, Hidden_Size=32, Learning_Rate=0.003

LSTM Hyperparameter Tuning Results:


,Lookback,Hidden_Size,Learning_Rate,MAE,RMSE,WAPE,R2,Validation_Folds
0,14,16,0.001,24.9624,30.2994,26.9607,-0.3110,22
1,28,16,0.001,25.0399,30.4504,27.0267,-0.3142,22
2,14,32,0.001,25.1970,30.3206,27.2479,-0.2823,22
3,14,32,0.003,25.3930,30.4472,27.5314,-0.2896,22
4,28,32,0.001,25.4730,30.3138,27.6170,-0.2904,22
5,14,16,0.003,25.5026,30.5012,27.6648,-0.3043,22
6,28,32,0.003,25.6945,30.6961,27.9159,-0.3264,22
7,28,16,0.003,25.7214,30.7292,27.9520,-0.3333,22



Best LSTM Configuration:
Lookback: 14
Hidden Units: 16
Learning Rate: 0.001
Best Tuned LSTM MAE: 24.9624

Current Overall Best Model:
Model: Weighted Averaging
MAE: 24.8586


#### Key Observations
- **Shorter Historical Lookback Improves LSTM Accuracy:** The best configuration uses a 14-day lookback, 16 hidden units, and a learning rate of 0.001, achieving an MAE of 24.96 and RMSE of 30.30 across 22 validation folds.
- **Weighted Averaging Retains the Overall Lead:** The tuned LSTM improves the original LSTM MAE from 25.04 to 24.96, but Weighted Averaging remains slightly better at 24.86.

#### Tuning Extra Trees Hyperparameters

In [40]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import ParameterSampler

extra_trees_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 8, 12, 16],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.7, 0.8, 1.0]
}

parameter_combinations = list(
    ParameterSampler(
        extra_trees_param_grid,
        n_iter=8,
        random_state=RANDOM_STATE
    )
)

extra_trees_tuning_results = []
best_tuning_mae = float("inf")
best_tuning_results_df = None
best_tuning_predictions_df = None
best_extra_trees_params = None

for config_number, params in enumerate(parameter_combinations, start=1):
    fold_results = []
    fold_predictions = []

    for _, fold in validation_folds_df.iterrows():
        fold_number = int(fold["fold"])

        fold_train = train_features_df[
            train_features_df[date_column] <= fold["train_end"]
        ]

        fold_validation = train_features_df[
            train_features_df[date_column].between(
                fold["validation_start"],
                fold["validation_end"]
            )
        ]

        if len(fold_validation) != forecast_horizon:
            raise ValueError(
                "Validation window must contain seven observations."
            )

        X_train_fold = fold_train[feature_columns]
        y_train_fold = fold_train[target_column]

        X_validation_fold = fold_validation[feature_columns]
        y_validation_fold = fold_validation[target_column]

        model = ExtraTreesRegressor(
            **params,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(X_train_fold, y_train_fold)

        predicted = np.maximum(
            model.predict(X_validation_fold),
            0
        )

        actual = y_validation_fold.to_numpy()
        metrics = evaluate_forecast(actual, predicted)

        fold_results.append({
            "Model": "Tuned Extra Trees",
            "Fold": fold_number,
            **metrics
        })

        for step, (actual_value, predicted_value) in enumerate(
            zip(actual, predicted),
            start=1
        ):
            fold_predictions.append({
                "Model": "Tuned Extra Trees",
                "Fold": fold_number,
                "Forecast_Step": step,
                "Actual": float(actual_value),
                "Predicted": float(predicted_value)
            })

    fold_results_df = pd.DataFrame(fold_results)
    fold_predictions_df = pd.DataFrame(fold_predictions)

    average_metrics = fold_results_df[
        ["MAE", "RMSE", "WAPE", "R2"]
    ].mean()

    extra_trees_tuning_results.append({
        "Configuration": config_number,
        **params,
        **average_metrics.to_dict(),
        "Validation_Folds": len(fold_results_df)
    })

    if average_metrics["MAE"] < best_tuning_mae:
        best_tuning_mae = average_metrics["MAE"]
        best_extra_trees_params = params.copy()
        best_tuning_results_df = fold_results_df.copy()
        best_tuning_predictions_df = fold_predictions_df.copy()

    print(
        f"Completed Configuration {config_number}/8 | "
        f"MAE: {average_metrics['MAE']:.4f}"
    )

extra_trees_tuning_df = (
    pd.DataFrame(extra_trees_tuning_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

extra_trees_tuning_df.to_csv(
    artifacts_dir / "extra_trees_hyperparameter_tuning.csv",
    index=False
)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Tuned Extra Trees"
        ],
        best_tuning_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Tuned Extra Trees"
        ],
        best_tuning_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("\nExtra Trees Hyperparameter Tuning Results:")
display(extra_trees_tuning_df.round(4))

print("\nBest Extra Trees Configuration:")
print(best_extra_trees_params)
print(f"Best Tuned Extra Trees MAE: {best_tuning_mae:.4f}")

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

Completed Configuration 1/8 | MAE: 25.8962
Completed Configuration 2/8 | MAE: 27.7629
Completed Configuration 3/8 | MAE: 27.4754
Completed Configuration 4/8 | MAE: 26.5567
Completed Configuration 5/8 | MAE: 25.6465
Completed Configuration 6/8 | MAE: 25.9762
Completed Configuration 7/8 | MAE: 28.0536
Completed Configuration 8/8 | MAE: 27.0533

Extra Trees Hyperparameter Tuning Results:


,Configuration,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,MAE,RMSE,WAPE,R2,Validation_Folds
0,5,100,2,4,0.7,12.0,25.6465,30.7887,28.0721,-0.4364,22
1,1,100,10,4,0.8,8.0,25.8962,31.1066,28.3212,-0.4479,22
2,6,100,2,4,0.8,8.0,25.9762,31.3446,28.4327,-0.4684,22
3,4,100,2,2,0.7,NaN,26.5567,32.0438,29.0810,-0.5402,22
4,8,200,10,1,0.8,12.0,27.0533,32.6164,29.5578,-0.5972,22
5,3,300,2,1,1.0,8.0,27.4754,33.3641,29.9888,-0.6958,22
6,2,100,2,1,0.8,8.0,27.7629,33.5414,30.3119,-0.7212,22
7,7,100,5,1,1.0,12.0,28.0536,34.1333,30.7061,-0.9308,22



Best Extra Trees Configuration:
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.7, 'max_depth': 12}
Best Tuned Extra Trees MAE: 25.6465

Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Weighted Averaging,24.8586,29.5802,27.0050,-0.2174,22
1,LSTM,25.0399,30.4504,27.0267,-0.3142,22
2,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
3,Tuned Extra Trees,25.6465,30.7887,28.0721,-0.4364,22
4,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
5,GRU,25.6713,30.8085,27.8006,-0.3373,22
6,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
7,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
8,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22
9,Extra Trees,25.7271,31.0040,28.1458,-0.4313,22



Total Models Evaluated: 26
Best Performing Model: Weighted Averaging
Best Model MAE: 24.8586


#### Key Observations
- **Hyperparameter Tuning Slightly Improves Extra Trees Accuracy:** The best configuration uses 100 estimators, a maximum depth of 12, a minimum leaf size of 4, and 70% feature sampling, reducing MAE from 25.73 to 25.65 across 22 validation folds.
- **Weighted Averaging Remains the Strongest Forecasting Approach:** Despite improvements in Extra Trees and LSTM, Weighted Averaging retains the lowest validation MAE of 24.86, demonstrating the value of combining complementary forecasting models.

#### Tuning SARIMA Hyperparameters

In [41]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_configurations = [
    {"order": (1, 0, 1), "seasonal_order": (1, 0, 0, 7)},
    {"order": (1, 1, 1), "seasonal_order": (1, 0, 0, 7)},
    {"order": (2, 0, 1), "seasonal_order": (1, 0, 0, 7)},
    {"order": (1, 0, 1), "seasonal_order": (1, 1, 0, 7)}
]

sarima_tuning_results = []
best_sarima_mae = float("inf")
best_sarima_params = None
best_sarima_results_df = None
best_sarima_predictions_df = None

for config_number, params in enumerate(sarima_configurations, start=1):
    fold_results = []
    fold_predictions = []

    for _, fold in validation_folds_df.iterrows():
        fold_number = int(fold["fold"])

        fold_train = train_df[
            train_df[date_column] <= fold["train_end"]
        ]

        fold_validation = train_df[
            train_df[date_column].between(
                fold["validation_start"],
                fold["validation_end"]
            )
        ]

        train_values = fold_train[target_column].to_numpy(dtype=float)
        actual = fold_validation[target_column].to_numpy(dtype=float)

        if len(actual) != forecast_horizon:
            raise ValueError(
                "Validation window must contain seven observations."
            )

        model = SARIMAX(
            train_values,
            order=params["order"],
            seasonal_order=params["seasonal_order"],
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted_model = model.fit(disp=False, maxiter=100)

        predicted = np.maximum(
            np.asarray(
                fitted_model.forecast(steps=forecast_horizon),
                dtype=float
            ),
            0
        )

        metrics = evaluate_forecast(actual, predicted)

        fold_results.append({
            "Model": "Tuned SARIMA",
            "Fold": fold_number,
            **metrics
        })

        for step, (actual_value, predicted_value) in enumerate(
            zip(actual, predicted),
            start=1
        ):
            fold_predictions.append({
                "Model": "Tuned SARIMA",
                "Fold": fold_number,
                "Forecast_Step": step,
                "Actual": float(actual_value),
                "Predicted": float(predicted_value)
            })

    fold_results_df = pd.DataFrame(fold_results)
    fold_predictions_df = pd.DataFrame(fold_predictions)

    average_metrics = fold_results_df[
        ["MAE", "RMSE", "WAPE", "R2"]
    ].mean()

    sarima_tuning_results.append({
        "Configuration": config_number,
        "Order": str(params["order"]),
        "Seasonal_Order": str(params["seasonal_order"]),
        **average_metrics.to_dict(),
        "Validation_Folds": len(fold_results_df)
    })

    if average_metrics["MAE"] < best_sarima_mae:
        best_sarima_mae = average_metrics["MAE"]
        best_sarima_params = params.copy()
        best_sarima_results_df = fold_results_df.copy()
        best_sarima_predictions_df = fold_predictions_df.copy()

    print(
        f"Completed Configuration {config_number}/4 | "
        f"MAE: {average_metrics['MAE']:.4f}"
    )

sarima_tuning_df = (
    pd.DataFrame(sarima_tuning_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

sarima_tuning_df.to_csv(
    artifacts_dir / "sarima_hyperparameter_tuning.csv",
    index=False
)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Tuned SARIMA"
        ],
        best_sarima_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Tuned SARIMA"
        ],
        best_sarima_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("\nSARIMA Hyperparameter Tuning Results:")
display(sarima_tuning_df.round(4))

print("\nBest SARIMA Configuration:")
print(best_sarima_params)
print(f"Best Tuned SARIMA MAE: {best_sarima_mae:.4f}")

print("\nUpdated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

Completed Configuration 1/4 | MAE: 25.5914
Completed Configuration 2/4 | MAE: 25.3564


d:\Projects\ML and Data Science\StaySense Analytics\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


Completed Configuration 3/4 | MAE: 25.6357
Completed Configuration 4/4 | MAE: 28.9166

SARIMA Hyperparameter Tuning Results:


,Configuration,Order,Seasonal_Order,MAE,RMSE,WAPE,R2,Validation_Folds
0,2,"(1, 1, 1)","(1, 0, 0, 7)",25.3564,30.0178,27.4934,-0.2436,22
1,1,"(1, 0, 1)","(1, 0, 0, 7)",25.5914,30.2074,27.7835,-0.2612,22
2,3,"(2, 0, 1)","(1, 0, 0, 7)",25.6357,30.2816,27.8367,-0.2760,22
3,4,"(1, 0, 1)","(1, 1, 0, 7)",28.9166,35.7389,31.6642,-1.2309,22



Best SARIMA Configuration:
{'order': (1, 1, 1), 'seasonal_order': (1, 0, 0, 7)}
Best Tuned SARIMA MAE: 25.3564

Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Weighted Averaging,24.8586,29.5802,27.0050,-0.2174,22
1,LSTM,25.0399,30.4504,27.0267,-0.3142,22
2,Tuned SARIMA,25.3564,30.0178,27.4934,-0.2436,22
3,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
4,Tuned Extra Trees,25.6465,30.7887,28.0721,-0.4364,22
5,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
6,GRU,25.6713,30.8085,27.8006,-0.3373,22
7,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
8,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22
9,Simple Exponential Smoothing,25.6880,30.1320,27.8631,-0.2483,22



Total Models Evaluated: 27
Best Performing Model: Weighted Averaging
Best Model MAE: 24.8586


#### Key Observations
- **SARIMA Tuning Produces a Marginal Improvement:** The configuration SARIMA(1,1,1)(1,0,0,7) achieves an MAE of 25.36 and RMSE of 30.02 across 22 validation folds, slightly improving the original SARIMA results.
- **Weighted Averaging Maintains the Lowest Validation Error:** With an MAE of 24.86, Weighted Averaging continues to outperform the individually tuned models, supporting its selection as the leading ensemble candidate.

#### Registering the Tuned LSTM Model

In [42]:
best_lstm_config = lstm_tuning_df.iloc[0]

lookback = int(best_lstm_config["Lookback"])
hidden_size = int(best_lstm_config["Hidden_Size"])
learning_rate = float(best_lstm_config["Learning_Rate"])

tuned_lstm_results = []
tuned_lstm_predictions = []

for _, fold in validation_folds_df.iterrows():
    fold_number = int(fold["fold"])

    fold_train = train_df[
        train_df[date_column] <= fold["train_end"]
    ]

    fold_validation = train_df[
        train_df[date_column].between(
            fold["validation_start"],
            fold["validation_end"]
        )
    ]

    train_values = fold_train[target_column].to_numpy(
        dtype=np.float32
    )

    actual = fold_validation[target_column].to_numpy(
        dtype=np.float32
    )

    if len(actual) != forecast_horizon:
        raise ValueError(
            "Validation window must contain seven observations."
        )

    scaler = StandardScaler()

    scaled_train = scaler.fit_transform(
        train_values.reshape(-1, 1)
    ).ravel()

    X_sequences = []
    y_sequences = []

    for start in range(
        len(scaled_train) - lookback - forecast_horizon + 1
    ):
        X_sequences.append(
            scaled_train[start:start + lookback]
        )

        y_sequences.append(
            scaled_train[
                start + lookback:
                start + lookback + forecast_horizon
            ]
        )

    X_tensor = torch.tensor(
        np.array(X_sequences),
        dtype=torch.float32
    ).unsqueeze(-1)

    y_tensor = torch.tensor(
        np.array(y_sequences),
        dtype=torch.float32
    )

    torch.manual_seed(RANDOM_STATE)

    model = LSTMForecaster(hidden_size=hidden_size)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0.001
    )

    criterion = nn.MSELoss()

    model.train()

    for epoch in range(100):
        optimizer.zero_grad()

        predictions = model(X_tensor)

        loss = criterion(predictions, y_tensor)
        loss.backward()

        optimizer.step()

    model.eval()

    input_sequence = torch.tensor(
        scaled_train[-lookback:],
        dtype=torch.float32
    ).reshape(1, lookback, 1)

    with torch.no_grad():
        scaled_forecast = (
            model(input_sequence)
            .numpy()
            .reshape(-1, 1)
        )

    predicted = np.maximum(
        scaler.inverse_transform(scaled_forecast).ravel(),
        0
    )

    metrics = evaluate_forecast(actual, predicted)

    tuned_lstm_results.append({
        "Model": "Tuned LSTM",
        "Fold": fold_number,
        **metrics
    })

    for step, (actual_value, predicted_value) in enumerate(
        zip(actual, predicted),
        start=1
    ):
        tuned_lstm_predictions.append({
            "Model": "Tuned LSTM",
            "Fold": fold_number,
            "Forecast_Step": step,
            "Actual": float(actual_value),
            "Predicted": float(predicted_value)
        })

tuned_lstm_results_df = pd.DataFrame(tuned_lstm_results)
tuned_lstm_predictions_df = pd.DataFrame(tuned_lstm_predictions)

model_results_df = pd.concat(
    [
        model_results_df[
            model_results_df["Model"] != "Tuned LSTM"
        ],
        tuned_lstm_results_df
    ],
    ignore_index=True
)

model_predictions_df = pd.concat(
    [
        model_predictions_df[
            model_predictions_df["Model"] != "Tuned LSTM"
        ],
        tuned_lstm_predictions_df
    ],
    ignore_index=True
)

model_comparison_df = (
    model_results_df
    .groupby("Model", as_index=False)
    .agg(
        MAE=("MAE", "mean"),
        RMSE=("RMSE", "mean"),
        WAPE=("WAPE", "mean"),
        R2=("R2", "mean"),
        Validation_Folds=("Fold", "nunique")
    )
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_results_df.to_csv(
    artifacts_dir / "model_validation_results.csv",
    index=False
)

model_predictions_df.to_csv(
    artifacts_dir / "model_validation_predictions.csv",
    index=False
)

model_comparison_df.to_csv(
    artifacts_dir / "model_comparison.csv",
    index=False
)

print("Updated Model Comparison:")
display(model_comparison_df.round(4))

best_model = model_comparison_df.iloc[0]

print(f"\nTotal Models Evaluated: {model_results_df['Model'].nunique()}")
print(f"Best Performing Model: {best_model['Model']}")
print(f"Best Model MAE: {best_model['MAE']:.4f}")

print("\nTuned LSTM Registration:")
print(f"Lookback: {lookback}")
print(f"Hidden Units: {hidden_size}")
print(f"Learning Rate: {learning_rate}")
print(f"Validation Folds: {len(tuned_lstm_results_df)}")
print(
    f"Validation MAE: "
    f"{tuned_lstm_results_df['MAE'].mean():.4f}"
)

Updated Model Comparison:


,Model,MAE,RMSE,WAPE,R2,Validation_Folds
0,Weighted Averaging,24.8586,29.5802,27.0050,-0.2174,22
1,Tuned LSTM,24.9624,30.2994,26.9607,-0.3110,22
2,LSTM,25.0399,30.4504,27.0267,-0.3142,22
3,Tuned SARIMA,25.3564,30.0178,27.4934,-0.2436,22
4,SARIMA,25.3638,30.1642,27.6425,-0.2890,22
5,Tuned Extra Trees,25.6465,30.7887,28.0721,-0.4364,22
6,ARIMA,25.6696,30.1731,27.8351,-0.2533,22
7,GRU,25.6713,30.8085,27.8006,-0.3373,22
8,Historical Moving Average,25.6809,30.4130,27.8670,-0.2786,22
9,Holt Linear Trend,25.6826,30.1377,27.8564,-0.2489,22



Total Models Evaluated: 28
Best Performing Model: Weighted Averaging
Best Model MAE: 24.8586

Tuned LSTM Registration:
Lookback: 14
Hidden Units: 16
Learning Rate: 0.001
Validation Folds: 22
Validation MAE: 24.9624


#### Key Observations
- **Tuned LSTM Results Are Successfully Registered:** The optimized 14-day lookback configuration achieves an MAE of 24.96 across 22 validation folds, improving upon the original LSTM while maintaining consistent evaluation coverage.
- **Weighted Averaging Remains the Leading Candidate:** Among the 28 evaluated configurations, Weighted Averaging records the lowest validation MAE of 24.86, making it the primary candidate for final testing.

#### Saving Selected Model Configurations

In [43]:
import json

selected_model_configurations = {
    "forecast_horizon": forecast_horizon,
    "random_state": RANDOM_STATE,
    "selection_metric": "MAE",
    "validation_folds": len(validation_folds_df),
    "primary_model": "Weighted Averaging",
    "selected_models": [
        "Weighted Averaging",
        "Tuned LSTM",
        "Tuned SARIMA",
        "Tuned Extra Trees"
    ],
    "tuned_lstm": {
        "lookback": int(best_lstm_config["Lookback"]),
        "hidden_size": int(best_lstm_config["Hidden_Size"]),
        "learning_rate": float(best_lstm_config["Learning_Rate"]),
        "weight_decay": 0.001,
        "epochs": 100
    },
    "tuned_sarima": {
        "order": list(best_sarima_params["order"]),
        "seasonal_order": list(best_sarima_params["seasonal_order"]),
        "enforce_stationarity": False,
        "enforce_invertibility": False,
        "maxiter": 100
    },
    "tuned_extra_trees": {
        **best_extra_trees_params,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "weighted_averaging": {
        "components": {
            "LSTM": 0.40,
            "SARIMA": 0.35,
            "Extra Trees": 0.25
        },
        "description": "Original model configurations used in validation"
    }
}

config_path = artifacts_dir / "selected_model_configurations.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(
        selected_model_configurations,
        file,
        indent=4
    )

print("Selected Model Configurations:")
print(json.dumps(selected_model_configurations, indent=4))

print(f"\nConfiguration Saved: {config_path}")
print(f"Total Selected Models: {len(selected_model_configurations['selected_models'])}")

Selected Model Configurations:
{
    "forecast_horizon": 7,
    "random_state": 42,
    "selection_metric": "MAE",
    "validation_folds": 22,
    "primary_model": "Weighted Averaging",
    "selected_models": [
        "Weighted Averaging",
        "Tuned LSTM",
        "Tuned SARIMA",
        "Tuned Extra Trees"
    ],
    "tuned_lstm": {
        "lookback": 14,
        "hidden_size": 16,
        "learning_rate": 0.001,
        "weight_decay": 0.001,
        "epochs": 100
    },
    "tuned_sarima": {
        "order": [
            1,
            1,
            1
        ],
        "seasonal_order": [
            1,
            0,
            0,
            7
        ],
        "enforce_stationarity": false,
        "enforce_invertibility": false,
        "maxiter": 100
    },
    "tuned_extra_trees": {
        "n_estimators": 100,
        "min_samples_split": 2,
        "min_samples_leaf": 4,
        "max_features": 0.7,
        "max_depth": 12,
        "random_state": 42,
        "n_

#### Key Observations
- **Model Development and Selection Are Complete:** Across 28 evaluated configurations, Weighted Averaging achieves the lowest validation MAE of 24.86, followed closely by Tuned LSTM at 24.96.
- **Selected Configurations Are Preserved for Reproducibility:** The ensemble weights and tuned model hyperparameters have been saved, enabling consistent model reconstruction and evaluation without repeating the tuning process.